# **Data Loading And Cleaning**

## **1. Setup**

This cell installs necessary libraries, imports modules, and downloads required data resources.

**Before running this cell, please follow Step 2 in the README to create and activate your Python virtual environment.**

**Note for macOS users:**  
If you encounter SSL errors (e.g., certificate verify failed) when downloading NLTK data, please refer to Step 3 in the README for instructions on how to resolve this.

In [1]:
%pip install -q accelerate torch transformers datasets scikit-learn pandas nltk ipywidgets beautifulsoup4 matplotlib seaborn wordcloud tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import re

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jasmi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jasmi\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jasmi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


## **2. Load Datasets**

In [4]:
# Deceptive Opinion Spam Corpus
hotel_reviews_df = pd.read_csv('../data/original/dosc_hotel_reviews.csv')
# Kaggle product reviews dataset
product_reviews_df = pd.read_csv('../data/original/kaggle_fake_reviews.csv')

# Basic check
print("Product Reviews:", product_reviews_df.shape)
print("Hotel Reviews:", hotel_reviews_df.shape)

Product Reviews: (40432, 4)
Hotel Reviews: (1600, 5)


## **3. Data Exploration**

### 3.1 Examine the structure of both datasets

In [ ]:
print("=== EXPLORING DATASETS ===")
print("\n1. Product Reviews Dataset (Kaggle):")
print(f"Shape: {product_reviews_df.shape}")
print(f"Columns: {product_reviews_df.columns.tolist()}")
print("\nFirst few rows:")
print(product_reviews_df.head())

print("\n2. Hotel Reviews Dataset (DOSC):")
print(f"Shape: {hotel_reviews_df.shape}")
print(f"Columns: {hotel_reviews_df.columns.tolist()}")
print("\nFirst few rows:")
print(hotel_reviews_df.head())

=== EXPLORING DATASETS ===

1. Product Reviews Dataset (Kaggle):
Shape: (40432, 4)
Columns: ['category', 'rating', 'label', 'text_']

First few rows:
             category  rating label  \
0  Home_and_Kitchen_5     5.0    CG   
1  Home_and_Kitchen_5     5.0    CG   
2  Home_and_Kitchen_5     5.0    CG   
3  Home_and_Kitchen_5     1.0    CG   
4  Home_and_Kitchen_5     5.0    CG   

                                               text_  
0  Love this!  Well made, sturdy, and very comfor...  
1  love it, a great upgrade from the original.  I...  
2  This pillow saved my back. I love the look and...  
3  Missing information on how to use it, but it i...  
4  Very nice set. Good quality. We have had the s...  

2. Hotel Reviews Dataset (DOSC):
Shape: (1600, 5)
Columns: ['deceptive', 'hotel', 'polarity', 'source', 'text']

First few rows:
  deceptive   hotel  polarity       source  \
0  truthful  conrad  positive  TripAdvisor   
1  truthful   hyatt  positive  TripAdvisor   
2  truthful   hya

### 3.2 Check for missing values

In [ ]:
print("\n=== MISSING VALUES CHECK ===")
print("Product Reviews missing values:")
print(product_reviews_df.isnull().sum())
print("\nHotel Reviews missing values:")
print(hotel_reviews_df.isnull().sum())


=== MISSING VALUES CHECK ===
Product Reviews missing values:
category    0
rating      0
label       0
text_       0
dtype: int64

Hotel Reviews missing values:
deceptive    0
hotel        0
polarity     0
source       0
text         0
dtype: int64


### 3.3 Check label distributions

In [ ]:
print("\n=== LABEL DISTRIBUTIONS ===")
print("Product Reviews labels:")
print(product_reviews_df['label'].value_counts())
print("\nHotel Reviews labels:")
print(hotel_reviews_df['deceptive'].value_counts())


=== LABEL DISTRIBUTIONS ===
Product Reviews labels:
label
CG    20216
OR    20216
Name: count, dtype: int64

Hotel Reviews labels:
deceptive
truthful     800
deceptive    800
Name: count, dtype: int64


### 3.4 Check text column names and sample texts

In [ ]:
print("\n=== TEXT SAMPLES ===")
print("Product Reviews text samples:")
text_col_product = [col for col in product_reviews_df.columns if 'text' in col.lower()][0]
print(f"Text column: {text_col_product}")
print(product_reviews_df[text_col_product].head(2).tolist())

print("\nHotel Reviews text samples:")
text_col_hotel = [col for col in hotel_reviews_df.columns if 'text' in col.lower()][0]
print(f"Text column: {text_col_hotel}")
print(hotel_reviews_df[text_col_hotel].head(2).tolist())


=== TEXT SAMPLES ===
Product Reviews text samples:
Text column: text_
['Love this!  Well made, sturdy, and very comfortable.  I love it!Very pretty', "love it, a great upgrade from the original.  I've had mine for a couple of years"]

Hotel Reviews text samples:
Text column: text
['We stayed for a one night getaway with family on a thursday. Triple AAA rate of 173 was a steal. 7th floor room complete with 44in plasma TV bose stereo, voss and evian water, and gorgeous bathroom(no tub but was fine for us) Concierge was very helpful. You cannot beat this location... Only flaw was breakfast was pricey and service was very very slow(2hours for four kids and four adults on a friday morning) even though there were only two other tables in the restaurant. Food was very good so it was worth the wait. I would return in a heartbeat. A gem in chicago... \n', 'Triple A rate with upgrade to view room was less than $200 which also included breakfast vouchers. Had a great view of river, lake, Wrigley

## **4.Data Preprocessing and Standardization**

### 4.1 Standardize datasets with raw text

In [ ]:
def standardize_dataset(df, text_col, label_col):
    """
    Standardize dataset format
    
    Args:
        df (pd.DataFrame): Input dataframe
        text_col (str): Name of text column
        label_col (str): Name of label column
        
    Returns:
        pd.DataFrame: Standardized dataframe
    """
    df_raw = df.copy()
    
    # Rename columns to standard names
    df_raw = df_raw.rename(columns={
        text_col: 'text',
        label_col: 'label'
    })
    
    # Keep only necessary columns
    df_raw = df_raw[['text', 'label']].copy()
    
    # Remove rows with missing text
    df_raw = df_raw.dropna(subset=['text'])
    
    return df_raw


In [ ]:
product_std_df = standardize_dataset(product_reviews_df, 'text_', 'label')
hotel_std_df = standardize_dataset(hotel_reviews_df, 'text', 'deceptive')

print("\n=== CLEANED DATASETS ===")
print("\n1. Product Reviews Dataset (Kaggle):")
print(product_std_df.head())
print("\n2. Hotel Reviews Dataset (DOSC):")
print(hotel_std_df.head())


=== CLEANED DATASETS ===

1. Product Reviews Dataset (Kaggle):
                                                text label
0  Love this!  Well made, sturdy, and very comfor...    CG
1  love it, a great upgrade from the original.  I...    CG
2  This pillow saved my back. I love the look and...    CG
3  Missing information on how to use it, but it i...    CG
4  Very nice set. Good quality. We have had the s...    CG

2. Hotel Reviews Dataset (DOSC):
                                                text     label
0  We stayed for a one night getaway with family ...  truthful
1  Triple A rate with upgrade to view room was le...  truthful
2  This comes a little late as I'm finally catchi...  truthful
3  The Omni Chicago really delivers on all fronts...  truthful
4  I asked for a high floor away from the elevato...  truthful


### 4.2 Convert labels to binary

In [ ]:
def convert_labels_to_binary(df, label_mapping=None):
    """Convert labels to binary format (0=real, 1=fake)"""
    df_binary = df.copy()
    
    # Default label mapping based on your data
    if label_mapping is None:
        label_mapping = {
            # Product reviews (Kaggle)
            'OR': 0,   # Original = real
            'CG': 1,   # Computer Generated = fake
            
            # Hotel reviews (DOSC)
            'truthful': 0,    # Real reviews
            'deceptive': 1,   # Fake reviews
            
            0: 0,
            1: 1,
            '0': 0,
            '1': 1,
        }
    
    # Apply mapping
    df_binary['label'] = df_binary['label'].map(label_mapping)
    
    # Remove rows with unmapped labels
    df_binary = df_binary.dropna(subset=['label'])
    df_binary['label'] = df_binary['label'].astype(int)
    
    return df_binary

In [ ]:
product_std_df = convert_labels_to_binary(product_std_df)
hotel_std_df = convert_labels_to_binary(hotel_std_df)

print("\n=== FINAL DATASETS ===")
print("\n1. Product Reviews Dataset (Kaggle):")
print(product_std_df.head())
print("\n2. Hotel Reviews Dataset (DOSC):")
print(hotel_std_df.head())


=== FINAL DATASETS ===

1. Product Reviews Dataset (Kaggle):
                                                text  label
0  Love this!  Well made, sturdy, and very comfor...      1
1  love it, a great upgrade from the original.  I...      1
2  This pillow saved my back. I love the look and...      1
3  Missing information on how to use it, but it i...      1
4  Very nice set. Good quality. We have had the s...      1

2. Hotel Reviews Dataset (DOSC):
                                                text  label
0  We stayed for a one night getaway with family ...      0
1  Triple A rate with upgrade to view room was le...      0
2  This comes a little late as I'm finally catchi...      0
3  The Omni Chicago really delivers on all fronts...      0
4  I asked for a high floor away from the elevato...      0


### 4.3 Clean the std dataste

In [ ]:
def clean_text(text):
    """
    Comprehensive text cleaning with HTML parsing, stopword removal, and lemmatization
    
    Args:
        text (str): Raw text to clean
        
    Returns:
        str: Cleaned text
    """
    if pd.isna(text):
        return ""
    
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters (keep letters, numbers, spaces)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Split into words
    words = text.split()
    
    # Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # Lemmatize words
    words = [lemmatizer.lemmatize(word) for word in words]
    
    # Join back into text
    return ' '.join(words)


product_clean_df = product_std_df.copy()
product_clean_df['text'] = product_std_df['text'].apply(clean_text)
product_clean_df = product_clean_df[product_clean_df['text'].str.len() > 0]

hotel_clean_df = hotel_std_df.copy()
hotel_clean_df['text'] = hotel_std_df['text'].apply(clean_text)
hotel_clean_df = hotel_clean_df[hotel_clean_df['text'].str.len() > 0]

In [ ]:
print("\n=== FINAL DATASETS ===")
print("\n1. Product Reviews Dataset (Kaggle):")
print(product_clean_df.head())
print("\n2. Hotel Reviews Dataset (DOSC):")
print(hotel_clean_df.head())


=== FINAL DATASETS ===

1. Product Reviews Dataset (Kaggle):
                                                text  label
0  love well made sturdy comfortable love itvery ...      1
1   love great upgrade original ive mine couple year      1
2            pillow saved back love look feel pillow      1
3        missing information use great product price      1
4                nice set good quality set two month      1

2. Hotel Reviews Dataset (DOSC):
                                                text  label
0  stayed one night getaway family thursday tripl...      0
1  triple rate upgrade view room less 200 also in...      0
2  come little late im finally catching review pa...      0
3  omni chicago really delivers front spaciousnes...      0
4  asked high floor away elevator got room pleasa...      0


## **5. Save Raw And Cleaned Data**

In [ ]:
import os

# Save the std raw datasets
product_std_df.to_csv('../data/processed/product_raw.csv', index=False)
hotel_std_df.to_csv('../data/processed/hotel_raw.csv', index=False)

print("✅ Standardized raw datasets saved to data/processed/")
print(f"✅ Product reviews: {product_std_df.shape}")
print(f"✅ Hotel reviews: {hotel_std_df.shape}")

✅ Standardized raw datasets saved to data/processed/
✅ Product reviews: (40432, 2)
✅ Hotel reviews: (1600, 2)


In [ ]:
# Save the cleaned datasets
product_clean_df.to_csv('../data/processed/product_clean.csv', index=False)
hotel_clean_df.to_csv('../data/processed/hotel_clean.csv', index=False)

print("✅ Cleaned datasets saved to data/processed/")
print(f"✅ Product reviews: {product_clean_df.shape}")
print(f"✅ Hotel reviews: {hotel_std_df.shape}")

✅ Cleaned datasets saved to data/processed/
✅ Product reviews: (40431, 2)
✅ Hotel reviews: (1600, 2)
